# Data Preprocessing

## Objective

This notebook performs all preprocessing steps required before model training.

Steps:

- Load raw dataset
- Clean target column
- Handle missing values
- Remove duplicates
- Remove unnecessary identifier/text columns
- Convert data types
- Feature engineering
- Train/Test Split
- Build preprocessing pipeline
- Save artifacts

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

os.makedirs("../artifacts", exist_ok=True)

In [2]:
df = pd.read_csv("../data/Patient_Health_Records_Raw.csv")

print(df.shape)
df.head()

(10000, 17)


,Patient_ID,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,ID-00001,"SMITH, ALICE",250,Unknown,Boston,22kg/m2,120 over 80,83,190,Unknown,yes,NaN,Apr 15 2021,30,A00,Patient feeling fine 🙂,0
1,ID-00002,"SMITH, ALICE",-5,FEMALE,newyork,25.8,NaN,500,normal,N,No,NaN,2021-07-10,NaN,unknown,"BP high, needs recheck",unknown
2,ID-00003,John Doe,-5,NaN,Nwe Yrok,22kg/m2,120 - 80,500,190,no,Former,"aspirin,metformin",Apr 06 2021,30,B20,NaN,unknown
3,ID-00004,michael brown,80,F,New York,33.4,NaN,eighty,250,y,EX-smoker,NaN,20241113,30,NaN,Follow up soon,unknown
4,ID-00005,Jane Doe,250,male,la,30.1,120 over 80,0,normal,no,No,NaN,2025-09-27,NaN,I10,Patient feeling fine 🙂,1


In [3]:
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
)

The target variable is **Has_Disease**.

The remaining columns will be treated as predictor variables.

In [4]:
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates()

print(df.shape)

Duplicates: 0
(10000, 17)


In [5]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [6]:
numeric_cols = [
    "Age",
    "BMI",
    "Blood_Pressure",
    "Heart_Rate",
    "Cholesterol_Level"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [7]:
df["Gender"] = (
    df["Gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "m": "male",
        "f": "female",
        "unknown": np.nan,
        "nan": np.nan
    })
)

In [8]:
df["City"] = (
    df["City"]
    .astype(str)
    .str.strip()
    .str.lower()
)

city_map = {
    "ny": "new york",
    "newyork": "new york",
    "nwe yrok": "new york",
    "la": "los angeles",
    "chicago": "chicago",
    "nan": np.nan
}

df["City"] = df["City"].replace(city_map)

In [9]:
df["Diagnosis_Code"] = (
    df["Diagnosis_Code"]
    .replace("unknown", np.nan)
)

In [10]:
df["Follow_Up"] = (
    df["Follow_Up"]
    .astype(str)
    .str.lower()
    .replace({
        "two weeks": "14",
        "nan": np.nan
    })
)

In [11]:
df["Medications"] = (
    df["Medications"]
    .astype(str)
    .str.lower()
    .replace({
        "aspirin,metformin": "aspirin + metformin",
        "aspirin; metformin": "aspirin + metformin",
        "lisinopril / metformin": "lisinopril + metformin",
        "nan": np.nan
    })
)

In [12]:
binary_map = {
    "yes": 1,
    "no": 0,
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0,
    "y": 1,
    "n": 0,
    "nan": np.nan,
    "unknown": np.nan
}

for col in ["Diabetic", "Smoker", "Has_Disease"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(binary_map)
    )

In [13]:
df = df.dropna(subset=["Has_Disease"])

df["Has_Disease"] = df["Has_Disease"].astype(int)

A stratified train-test split is used to preserve the distribution of the target classes in both training and testing datasets.

In [14]:
drop_columns = [
    "Patient_ID",
    "Name",
    "Last_Visit_Date",
    "Blood_Pressure",
    "Visit_Year",
    "Visit_Month",
    "Visit_Day",
    "Notes"
]

df.drop(columns=drop_columns, inplace=True, errors="ignore")

In [15]:
print(df.isnull().sum())

Age                  1228
Gender               1688
City                  540
BMI                  2429
Heart_Rate           1223
Cholesterol_Level    2976
Diabetic             1650
Smoker               3310
Medications          1986
Follow_Up            1272
Diagnosis_Code        826
Has_Disease             0
dtype: int64


In [16]:
X = df.drop("Has_Disease", axis=1)

y = df["Has_Disease"]

The preprocessing pipeline ensures that transformations are fitted only on the training data, preventing data leakage.

In [17]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['Age', 'BMI', 'Heart_Rate', 'Cholesterol_Level', 'Diabetic', 'Smoker']
Categorical: ['Gender', 'City', 'Medications', 'Follow_Up', 'Diagnosis_Code']


In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [19]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [20]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [21]:
preprocessor.fit(X)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'BMI', 'Heart_Rate',
                                  'Cholesterol_Level', 'Diabetic', 'Smoker']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'City', 'Medications', 'Follow_Up',
                                  'Diagnosis_Code'])])

In [22]:
clean_df = X.copy()
clean_df["Has_Disease"] = y

clean_df.to_csv(
    "../artifacts/clean_dataset.csv",
    index=False
)

In [23]:
joblib.dump(
    preprocessor,
    "../artifacts/preprocessor.joblib"
)

['../artifacts/preprocessor.joblib']

In [24]:
feature_names = preprocessor.get_feature_names_out()

joblib.dump(
    feature_names,
    "../artifacts/feature_names.joblib"
)

['../artifacts/feature_names.joblib']

In [25]:
print("Preprocessing completed successfully.")

print(clean_df.shape)

print("Features:", len(feature_names))

Preprocessing completed successfully.
(4974, 12)
Features: 21


# Completed

The dataset has now been:

- cleaned
- validated
- encoded
- scaled
- split
- saved

The preprocessing pipeline is reusable and leakage-safe.

The next notebook will train multiple machine learning models using cross-validation and automatically select the best-performing model.